# Agentic Search: live demonstration
**CIKM 2026 tutorial "Agentic Search: Foundations, Optimization, and Long-Horizons"** (Chuan Meng, Litu Ou, Sean MacAvaney, Jeff Dalton). Website: [agentic-search.github.io/cikm2026](https://agentic-search.github.io/cikm2026/)

This notebook runs a search agent over the corpus of [BrowseComp-Plus](https://github.com/texttron/BrowseComp-Plus), about 100K web pages. We wrote a few everyday questions that this corpus can answer and marked the pages that contain the answer, so we can watch not only whether the agent answers correctly but also when those pages show up in its search results. Everything runs on whole documents.

The demo walks through three findings of our SIGIR 2026 paper on the retrieval side of agentic search: the retriever matters, re-ranking helps, and reading whole documents helps.

### How to use this notebook
Run the first three cells (Setup, Helpers and Connect), then go through the sections in order. The notebook works in one of two modes.

**Replay mode (default, no setup).** If you leave the endpoints in the *Connect* cell empty, the notebook replays runs we recorded in advance, in the same viewer that we used in the tutorial. Sections 1 to 4 work in this mode; Section 5 (your own question) needs live mode.

**Live mode (your own endpoints).** To run the agent yourself, host two services and enter their addresses in the *Connect* cell:

1. **The agent model**: any OpenAI-compatible chat endpoint that supports tool calls. We used `gpt-oss-20b`, for example served with vLLM (`vllm serve openai/gpt-oss-20b --served-model-name gpt-oss-20b`), which gives a base URL such as `http://<your-host>:8000/v1`. The cells below ask for the model by the name `gpt-oss-20b`.
2. **The search tools**: a small HTTP service over the BrowseComp-Plus corpus with three routes:
   * `GET /health` returns `{"ok": true}`;
   * `POST /tools/search` takes `{"user_query": ..., "retriever": ..., "k": 5}` and returns `{"results": [{"docid": ..., "score": ..., "snippet": ...}, ...]}`, where the snippet is the first 512 tokens of the document;
   * `POST /tools/get_document` takes `{"docid": ...}` and returns `{"docid": ..., "text": ...}`.

   The `retriever` field is one of `bm25`, `colbert` and `bm25+monot5`, the settings compared in Sections 2 and 3. The corpus, pre-built indexes and retrieval code are available in the [BrowseComp-Plus repository](https://github.com/texttron/BrowseComp-Plus).

If the notebook runs on Colab, both services must be reachable from Colab's machine. Alternatively, run the services on your own machine and connect Colab to it with *Connect to a local runtime*. If an endpoint cannot be reached, the notebook falls back to replay mode.

In [1]:
#@title Setup: run this cell first { display-mode: "form" }
# Agentic Search tutorial, CIKM 2026: live demonstration notebook.
# This cell defines the client for the search endpoint, the trajectory viewer and the replay of recorded runs.
# The agent itself is in the next section, in plain sight.
import base64, hashlib, json, os, re, threading, time, uuid
import requests
try:
    from openai import OpenAI
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
    from openai import OpenAI
from IPython.display import HTML, Markdown, display

SEARCH_TOOL = {"type": "function", "function": {
    "name": "search",
    "description": "Perform a search on a knowledge source. Returns top-5 hits with docid, score, and snippet. "
                   "The snippet contains the document's contents (may be truncated based on token limits).",
    "parameters": {"type": "object", "properties": {"user_query": {
        "type": "string", "description": "Query to search the local knowledge base for relevant information"}},
        "required": ["user_query"], "additionalProperties": False}}}
GET_DOCUMENT_TOOL = {"type": "function", "function": {
    "name": "get_document", "description": "Retrieve a full document by its docid.",
    "parameters": {"type": "object", "properties": {"docid": {"type": "string", "description": "Document ID to retrieve"}},
                   "required": ["docid"], "additionalProperties": False}}}


class DemoServer:
    """Thin client for the search endpoint (search, get_document). Our demo question and its grading run locally."""

    def __init__(self, base_url, api_key, timeout=120, local_questions=None):
        self.base, self.h, self.timeout = base_url.rstrip("/"), {"Authorization": f"Bearer {api_key}"}, timeout
        # our own questions can ship with the notebook: {qid: {"question", "answer", "aliases", "evidence_docids"}}; question()
        # and check() answer them here (graded by word match: every word of the gold answer or of an alias appears in the
        # Exact Answer)
        self.local = dict(local_questions or {})

    def _post(self, path, body):
        r = requests.post(self.base + path, json=body, headers=self.h, timeout=self.timeout)
        if r.status_code != 200:
            raise RuntimeError(f"{path}: HTTP {r.status_code} {r.text[:300]}")
        return r.json()

    def _get(self, path):
        r = requests.get(self.base + path, headers=self.h, timeout=self.timeout)
        r.raise_for_status()
        return r.json()

    def search(self, query, retriever="bm25", k=5, reasoning=""):
        """Returns the full reply: {"results": [...], "q2q": the rewritten question when the retriever uses Q2Q, ...}."""
        return self._post("/tools/search", {"user_query": query, "retriever": retriever, "k": k, "reasoning": reasoning})

    def get_document(self, docid):
        return self._post("/tools/get_document", {"docid": docid})

    def question(self, qid):
        q = self.local.get(str(qid))
        if q:
            return {"qid": str(qid), "question": q["question"], "evidence_docids": list(q["evidence_docids"]),
                    "gold_docids": list(q["evidence_docids"])}
        return self._post("/bcp/question", {"qid": str(qid)})

    def prompts(self):
        return self._get("/bcp/prompts")

    def lab(self):
        return self._get("/bcp/lab")

    def check(self, qid, response):
        q = self.local.get(str(qid))
        if q:
            norm = lambda x: set(re.sub(r"[^0-9a-z ]+", " ", str(x).lower()).split())
            got = norm(exact_answer(response))
            ok = bool(got) and any(norm(g) <= got for g in [q["answer"]] + list(q.get("aliases", [])))
            gold = q.get("display_answer") or (q["answer"] + "".join(f" (or {x})" for x in q.get("aliases", [])))
            return {"qid": str(qid), "correct": ok, "gold_answer": gold, "extracted_answer": exact_answer(response),
                    "evidence_docids": list(q["evidence_docids"]), "gold_docids": list(q["evidence_docids"]),
                    "judge": "word match against the gold answer"}
        return self._post("/bcp/check", {"qid": str(qid), "response": response})

    def report(self, **kw):
        try:
            self._post("/bcp/report", kw)
        except Exception:  # noqa: BLE001  (the results page is optional)
            pass


def clean_tool_name(name, offered):
    """gpt-oss served by vLLM sometimes leaks Harmony channel tokens into the tool name (e.g. 'search<|channel|>commentary'
    or 'functions.search'); map such a name back to the tool that was offered."""
    clean = re.sub(r"<\|.*$", "", name or "").strip()
    clean = clean[len("functions."):] if clean.startswith("functions.") else clean
    return clean if clean in offered else next((t for t in sorted(offered, key=len, reverse=True) if clean.startswith(t)), clean)


def exact_answer(text):
    m = re.search(r"Exact Answer\W{0,4}:\W{0,4}\s*(.+)", text or "", re.I)
    return m.group(1).strip().strip("*").strip().replace("\u202f", " ") if m else ""



# ---------------------------------------------------------------- trajectory viewer
_CALL = re.compile(r'(?:functions\.)?(?:search|get_document)?\s*\(?\s*\{\s*"(?:user_query|query|docid)"\s*:[^\n]*?\}\s*\)?')


def clean_reasoning(text):
    """The reasoning as shown in the viewer: gpt-oss sometimes ends its reasoning with the tool call it is about to make
    ({"user_query": ...}); the viewer shows the query separately, so the copy inside the reasoning is dropped (Chuan 11-04).
    Only the display changes; the model still gets its full reasoning back."""
    out = []
    for line in (text or "").split("\n"):
        if _CALL.search(line):
            line = _CALL.sub("", line).rstrip()
            if not re.search(r"\w", line):
                continue
        out.append(line)
    return "\n".join(out).strip()

try:
    from IPython.display import HTML, display
except ImportError:  # rendering to a file still works without IPython
    HTML = display = None

PHASES = ["Reasoning", "Search query", "Search results", "Read document", "Answer"]   # the step strip (Read document only with get_document)

TEMPLATE = r"""
<div id="__ID__" class="tv" tabindex="0">
<style>
#__ID__{--ink:#1d1d1f;--mute:#6b6b70;--line:#d9d6cf;--paper:#fbfaf7;--blue:#1f5fbf;--warn:#a5361b;
  --gold:#9a6a00;--goldfill:#f2b705;--goldbg:#fff2c4;
  font-family:Calibri,"Segoe UI",Arial,sans-serif;font-size:17px;color:var(--ink);background:var(--paper);border:1px solid var(--line);
  padding:14px 16px 10px;line-height:1.35;max-width:1500px;outline:none}
#__ID__ *{box-sizing:border-box}
#__ID__ .qlab{font-size:13px;letter-spacing:.06em;text-transform:uppercase;color:var(--mute);font-weight:600;margin-bottom:6px}
#__ID__ .q{font-size:23px;font-weight:600;line-height:1.4;margin:0 0 22px;white-space:pre-wrap;max-width:1150px}
#__ID__ .loop{display:flex;align-items:stretch;flex-wrap:wrap;row-gap:10px;font-size:16px;margin:0 0 14px}
#__ID__ .loop>.lab{align-self:center;margin-right:14px}
#__ID__ .box{border:1.5px solid var(--line);background:#fff;padding:6px 12px 9px}
#__ID__ .box .bh{font-size:12.5px;letter-spacing:.06em;text-transform:uppercase;color:var(--mute);font-weight:600;margin-bottom:6px;white-space:nowrap}
#__ID__ .box .bh b{color:var(--ink);text-transform:none;letter-spacing:0;font-size:13.5px}
#__ID__ .box.llm{border-color:#a9bde0}
#__ID__ .box.tools{background:#f3f1ea;border-style:dashed;border-color:#aaa598}
#__ID__ .chips{display:flex;align-items:center;gap:10px;flex-wrap:wrap}
#__ID__ .ph{padding:4px 12px;border:1px solid var(--line);color:var(--mute);background:#fff;white-space:nowrap}
#__ID__ .ph code{font-family:Menlo,Consolas,monospace;font-size:14.5px}
#__ID__ .ph small{font-size:13.5px;margin-left:6px}
#__ID__ .ph.on{border-color:var(--blue);background:var(--blue);color:#fff}
#__ID__ .ph.on.busy{background:#fff;color:var(--blue);border-style:dashed}
#__ID__ .ph.off{background:transparent;border-style:dashed;opacity:.6}
#__ID__ .ph.off code{text-decoration:line-through}
#__ID__ .then{font-size:14px;color:var(--mute);white-space:nowrap}
#__ID__ .wire{display:flex;flex-direction:column;justify-content:center;gap:8px;padding:12px 14px 0;min-width:120px;font-size:14.5px;color:var(--mute);white-space:nowrap}
#__ID__ .wire span{border-bottom:1.5px solid var(--line);padding:0 2px 1px;text-align:center}
#__ID__ .wire span.on{color:var(--blue);font-weight:700;border-color:var(--blue)}
#__ID__ .ctl{display:flex;align-items:center;gap:10px;row-gap:8px;flex-wrap:wrap;font-size:15.5px;color:var(--mute);margin:0 0 18px}
#__ID__ .ctl .info{margin-left:6px}
#__ID__ .ctl button{font:inherit;font-size:15.5px;padding:4px 14px;border:1px solid var(--line);background:#fff;color:var(--ink);cursor:pointer}
#__ID__ .ctl button.main{background:var(--blue);border-color:var(--blue);color:#fff;font-weight:700}
#__ID__ .ctl button:disabled{opacity:.45;cursor:default}
#__ID__ .ctl button.live.on{background:var(--ink);border-color:var(--ink);color:#fff}
#__ID__ .meta{display:flex;justify-content:space-between;gap:24px;row-gap:6px;flex-wrap:wrap;font-size:15.5px;color:var(--mute);margin-bottom:10px;
  padding-top:12px;border-top:1px solid var(--line)}
#__ID__ .meta .grp{display:flex;gap:22px;flex-wrap:wrap}
#__ID__ .meta b{color:var(--ink);font-weight:600}
#__ID__ .ctx{display:flex;align-items:center;gap:12px;flex-wrap:wrap;font-size:15px;color:var(--mute);margin:0 0 16px}
#__ID__ .bar{flex:1;min-width:200px;max-width:620px;height:12px;background:#e7e4dc;border:1px solid var(--line);position:relative}
#__ID__ .bar i{position:absolute;left:0;top:0;bottom:0;background:var(--blue)}
#__ID__ .bar i.warn{background:#c98a12} #__ID__ .bar i.full{background:var(--warn)}
#__ID__ .body{display:grid;grid-template-columns:minmax(260px,36%) 1fr;gap:0;border-top:1px solid var(--line);height:600px}
#__ID__ .list{overflow:auto;border-right:1px solid var(--line);padding:0 0 4px}
#__ID__ .cap{position:sticky;top:0;z-index:1;background:var(--paper);border-bottom:1px solid var(--line);padding:7px 10px 7px 12px;
  font-size:14px;color:var(--mute);line-height:1.3}
#__ID__ .cap b{color:var(--ink);text-transform:uppercase;letter-spacing:.05em;font-size:13px;margin-right:6px}
#__ID__ .det .cap{margin:0 -16px 6px;padding-left:16px}
#__ID__ .lab{font-size:13px;letter-spacing:.05em;text-transform:uppercase;color:var(--mute);font-weight:600;margin-right:4px}
#__ID__ .row{padding:7px 10px 7px 8px;cursor:pointer;border-left:4px solid transparent;display:grid;grid-template-columns:34px 1fr;gap:4px}
#__ID__ .row:hover{background:#f1efe9}
#__ID__ .row.sel{background:#e9eef8;border-left-color:var(--blue)}
#__ID__ .row .n{color:var(--mute);font-variant-numeric:tabular-nums}
#__ID__ .row .qq{font-family:Menlo,Consolas,monospace;font-size:14.5px;word-break:break-word}
#__ID__ .row .sub{font-size:14px;color:var(--mute);margin-top:3px}
#__ID__ .row.pend .qq,#__ID__ .row .qq.pend{color:var(--mute);font-style:italic;font-family:inherit}
#__ID__ .row.ans .qq{font-family:inherit;font-weight:700}
#__ID__ .rk{display:inline-block;min-width:21px;height:21px;line-height:19px;text-align:center;font-size:12.5px;margin-right:3px;
  background:#fff;color:var(--mute);border:1px solid var(--line);font-family:Calibri,"Segoe UI",Arial,sans-serif}
#__ID__ .rk.on{background:var(--goldfill);border-color:var(--goldfill);color:#1d1d1f;font-weight:700}
#__ID__ .det{overflow:auto;padding:0 16px 16px;position:relative}
#__ID__ h4{margin:10px 0 4px;font-size:15px;text-transform:uppercase;letter-spacing:.04em;color:var(--mute);font-weight:600}
#__ID__ .rs{white-space:pre-wrap;font-size:16px;background:#fff;border:1px solid var(--line);padding:8px 10px;max-height:300px;overflow:auto}
#__ID__ .query{font-family:Menlo,Consolas,monospace;font-size:17px;color:var(--blue);background:#fff;border-left:4px solid var(--blue);padding:6px 10px}
#__ID__ .q2q{font-size:16px;color:var(--ink);background:#fff;border-left:4px solid var(--mute);padding:5px 10px;margin-top:4px}
#__ID__ .q2q span{color:var(--mute);font-size:14px}
#__ID__ .waiting{color:var(--mute);font-style:italic;margin:6px 0}
#__ID__ table{border-collapse:collapse;width:100%;font-size:15.5px}
#__ID__ td{border-bottom:1px solid var(--line);padding:6px 6px;vertical-align:top}
#__ID__ td.r{width:34px;padding-top:7px}
#__ID__ td.d{font-family:Menlo,Consolas,monospace;font-size:14px;width:74px}
#__ID__ tr.rel td{background:var(--goldbg)}
#__ID__ tr.rel td:first-child{box-shadow:inset 6px 0 0 var(--goldfill)}
#__ID__ tr.rel b{color:var(--ink)}
#__ID__ .badge{display:inline-block;font-size:12.5px;font-weight:700;color:#1d1d1f;background:var(--goldfill);padding:1px 6px;margin-left:8px;vertical-align:1px;letter-spacing:.03em}
#__ID__ details summary{cursor:pointer;color:var(--mute);font-size:14px}
#__ID__ .snip{white-space:pre-wrap;font-size:14px;color:#333;max-height:220px;overflow:auto;background:#fff;padding:6px;border:1px solid var(--line)}
#__ID__ .exact{font-size:26px;font-weight:700;margin:4px 0}
#__ID__ .ok{color:#11795a;font-weight:700}
#__ID__ .bad{color:var(--warn);font-weight:700}
#__ID__ .cite{font-family:Menlo,Consolas,monospace;font-size:14px;padding:0 3px;border:1px solid var(--line)}
#__ID__ .cite.g{background:var(--goldfill);color:#1d1d1f;font-weight:700}
#__ID__ .foot{border-top:1px solid var(--line);padding-top:8px;font-size:15px}
#__ID__ .ev span.d{display:inline-block;margin:2px 10px 2px 0;font-family:Menlo,Consolas,monospace;font-size:13.5px;padding:1px 6px}
#__ID__ .ev .hit{background:var(--goldfill);color:#1d1d1f;font-weight:700}
#__ID__ .ev .miss{color:var(--mute)}
#__ID__ .err{color:var(--warn)}
</style>
<div class="qlab"></div>
<div class="q"><span class="qt"></span></div>
<div class="loop"></div>
<div class="ctl"></div>
<div class="meta"></div>
<div class="ctx"></div>
<div class="body"><div class="list"></div><div class="det"></div></div>
<div class="foot"><div class="ev"></div></div>
<script>
(function(){
const S = __DATA__;
const ROOT_ID = "__ID__";
const esc = s => String(s ?? "").replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const pd = d => String(d ?? "").startsWith("browsecomp_plus_") ? String(d).slice(16).replace(/_[0-9]+$/, "") : String(d ?? "");
const D = r => pd(r.doc || r.docid);
const EV = new Set(S.evidence || []);
const RET = {"bm25": "BM25", "bm25-default": "BM25 (default k1 0.9, b 0.4)", "bm25-passage": "BM25 over passages",
             "colbert": "ColBERTv2", "colbert+q2q": "ColBERTv2 + Q2Q", "bm25+monot5": "BM25 top 50 → monoT5",
             "bm25+monot5+q2q": "BM25 top 50 → monoT5 + Q2Q"};

// ---- per-run viewer state survives re-renders (window first, sessionStorage as a fallback)
const G = (window.__tv = window.__tv || {});
let st = G[S.run];
if (!st) { try { st = JSON.parse(sessionStorage.getItem("tv-" + S.run) || "null"); } catch (e) {} }
if (!st) st = {k: 0, follow: !S.step_mode, sel: null};
G[S.run] = st;
const save = () => { try { sessionStorage.setItem("tv-" + S.run, JSON.stringify(st)); } catch (e) {} };

// ---- the run as a list of steps: Reasoning, Search query, Search results (or Read document), ..., Reasoning, Answer
const steps = [];
S.turns.forEach((t, i) => {
  steps.push({ph: "Reasoning", i, t: t.t});
  if (t.actions.length) t.actions.forEach((a, j) => {
    if (a.type === "search") { steps.push({ph: "Search query", i, j, t: t.t}); steps.push({ph: "Search results", i, j, t: a.t ?? t.t}); }
    else steps.push({ph: "Read document", i, j, t: a.t ?? t.t});
  });
  else if ((t.calls || []).length) steps.push({ph: t.calls[0].name === "get_document" ? "Read document" : "Search query", i, j: 0, wait: true, t: t.t});
});
if (S.answer) {
  steps.push({ph: "Reasoning", i: S.turns.length, fin: true, t: S.answer.t});
  steps.push({ph: "Answer", i: S.turns.length, t: S.answer.t});
} else if (S.status === "running" && S.pending === "model") steps.push({ph: "Reasoning", i: S.turns.length, pend: true, t: S.elapsed});
const N = steps.length;
const idx = {}; steps.forEach((s, n) => { idx[s.ph + "|" + s.i + "|" + (s.j ?? "")] = n; });
const shown = (ph, i, j) => { const n = idx[ph + "|" + i + "|" + (j ?? "")]; return n !== undefined && n <= st.k; };

function act(what) {
  if (what === "next") {
    const c = steps[st.k];
    if (st.sel !== null && c && st.sel !== c.i) { st.sel = null; save(); paint(); return; }   // first Next returns from an earlier turn
    if (st.k < N - 1) st.k++; st.follow = false; }
  else if (what === "back") { st.follow = false; if (st.k > 0) st.k--; }
  else if (what === "live") { st.follow = !st.follow; }
  else if (what === "first") { st.follow = false; st.k = 0; }
  st.sel = null; save(); paint();
}
window.__tvAct = window.__tvAct || {}; window.__tvAct[S.run] = act;
if (S.step_mode && !window.__tvActive) window.__tvActive = S.run;
if (!window.__tvKeys) {           // presenter keys: right/PageDown = next, left/PageUp = back, L = live
  window.__tvKeys = true;
  document.addEventListener("mousedown", e => {
    const v = e.target.closest && e.target.closest(".tv");
    window.__tvActive = v ? v.dataset.run : null;
  }, true);
  document.addEventListener("keydown", e => {
    const run = window.__tvActive, f = run && window.__tvAct && window.__tvAct[run];
    if (!f || e.altKey || e.ctrlKey || e.metaKey) return;
    const t = e.target; if (t && (t.isContentEditable || /INPUT|TEXTAREA|SELECT/.test(t.tagName) || (t.closest && t.closest(".CodeMirror,.cm-editor,.monaco-editor")))) return;
    const m = {"ArrowRight": "next", "PageDown": "next", "ArrowLeft": "back", "PageUp": "back", "l": "live", "Home": "first"}[e.key];
    if (m) { e.preventDefault(); f(m); }
  });
}

function paint() {
  const root = document.getElementById(ROOT_ID);
  if (!root) return;
  root.dataset.run = S.run;
  if (st.follow || N === 0) st.k = Math.max(0, N - 1);
  st.k = Math.max(0, Math.min(st.k, Math.max(0, N - 1)));
  const cur = steps[st.k] || {ph: "Reasoning", i: 0, pend: true, t: S.elapsed};
  const busy = !!(cur.pend || cur.wait);
  const hasDoc = (S.config.tools || "").includes("get_document");

  root.querySelector(".qlab").textContent = S.qid === "audience" ? "Question from the audience" :
    String(S.qid ?? "").startsWith("c") ? `Question ${S.qid} · written by us, answerable from the BrowseComp-Plus corpus` : `Question · BrowseComp-Plus #${S.qid ?? ""}`;
  root.querySelector(".qt").textContent = S.question;

  // the agent loop: the LLM on the left, the tools it may call (outside the LLM) on the right, the current step lit
  const turnNo = Math.min(cur.i + 1, S.turns.length + (S.answer ? 1 : 0)) || 1;
  const lit = (on, extra) => `ph ${on ? "on" : ""} ${on && busy ? "busy" : ""} ${extra || ""}`;
  const isS = cur.ph === "Search query" || cur.ph === "Search results", isD = cur.ph === "Read document";
  const callOn = cur.ph === "Search query" || isD, resOn = (cur.ph === "Search results" || isD) && !busy;
  root.querySelector(".loop").innerHTML = `<span class="lab">Agent loop</span>` +
    `<div class="box llm"><div class="bh">The LLM &middot; <b>${esc(S.config.model || "")}</b> &middot; turn ${turnNo}</div><div class="chips">` +
      `<span class="${lit(cur.ph === "Reasoning")}">Reasoning${cur.ph === "Reasoning" && busy ? "&hellip;" : ""}</span>` +
      `<span class="${lit(cur.ph === "Answer")}">Answer</span></div></div>` +
    `<div class="wire"><span class="${callOn ? "on" : ""}">${isD ? "docid" : "query"} &#10230;</span><span class="${resOn ? "on" : ""}">&#10229; ${isD ? "page" : "results"}</span></div>` +
    `<div class="box tools"><div class="bh">Tools, outside the LLM</div><div class="chips">` +
      `<span class="${lit(isS)}"><code>search(query)</code><small>&rarr; top 5 pages</small></span>` +
      (hasDoc ? `<span class="${lit(isD)}"><code>get_document(docid)</code><small>&rarr; full page</small></span>`
              : `<span class="ph off"><code>get_document(docid)</code><small>not given in this run</small></span>`) +
    `</div></div>`;

  // controls
  const running = S.status === "running";
  const ahead = N - 1 - st.k;
  const info = st.follow ? (running ? "following the agent live" : "") :
    (ahead > 0 ? `step ${st.k + 1} of ${N}${running ? " · agent is " + ahead + " step" + (ahead > 1 ? "s" : "") + " ahead" : ""}` :
     running ? "waiting for the agent&hellip;" : `step ${st.k + 1} of ${N}`);
  root.querySelector(".ctl").innerHTML =
    `<button data-a="back" ${st.k <= 0 ? "disabled" : ""}>&#9664; Back</button>` +
    `<button class="main" data-a="next" ${st.k >= N - 1 ? "disabled" : ""}>Next &#9654;</button>` +
    `<button class="live ${st.follow ? "on" : ""}" data-a="live">Live</button><span class="info">${info}</span>`;
  root.querySelectorAll(".ctl button").forEach(b => b.addEventListener("click", e => { e.stopPropagation(); window.__tvActive = S.run; act(b.dataset.a); }));

  // counters up to the shown step
  const nS = steps.filter((s, n) => n <= st.k && s.ph === "Search query").length;
  const nD = steps.filter((s, n) => n <= st.k && s.ph === "Read document").length;
  const ctxTurn = Math.min(cur.i, S.turns.length - 1);
  const ctx = (cur.fin || cur.ph === "Answer") ? (S.answer.context_tokens || 0) : (ctxTurn >= 0 && !cur.pend ? S.turns[ctxTurn].context_tokens || 0 : (S.turns.length ? S.turns[S.turns.length - 1].context_tokens || 0 : 0));
  root.querySelector(".meta").innerHTML =
    `<span class="grp"><span>agent <b>${esc(S.config.model)}</b></span><span>retriever <b>${esc(RET[S.config.retriever] || S.config.retriever)}</b></span>` +
    `<span>tools <b>${esc(S.config.tools || "search")}</b></span></span>` +
    `<span class="grp"><span>searches <b>${nS}</b></span>` + (nD ? `<span>documents read <b>${nD}</b></span>` : "") +
    `<span>time <b>${Math.round(cur.t ?? S.elapsed)}s</b></span><span>${running ? "running" : S.status === "replay" ? "replay" : "finished"}</span></span>`;
  const LIM = S.context_limit || 131072, pct = Math.min(100, 100 * ctx / LIM);
  root.querySelector(".ctx").innerHTML = `<span>context</span><span class="bar"><i class="${pct > 85 ? "full" : pct > 60 ? "warn" : ""}" style="width:${pct.toFixed(1)}%"></i></span>` +
    `<span><b style="color:var(--ink)">${(ctx / 1000).toFixed(1)}k</b> of ${(LIM / 1000).toFixed(0)}k tokens used, ${((LIM - ctx) / 1000).toFixed(0)}k left</span>`;

  // left: one row per turn, only what has been shown
  const strip = a => `<div class="sub">` + (a.results || []).map((r, k) => `<span class="rk ${EV.has(D(r)) ? "on" : ""}">${k + 1}</span>`).join("") + `</div>`;
  const rows = [];
  const lastTurn = Math.min(cur.i, S.turns.length - 1);
  for (let i = 0; i <= lastTurn; i++) {
    const t = S.turns[i];
    let main = "", sub = "";
    if (t.actions.length) {
      main = t.actions.map((a, j) => a.type === "search" ? (shown("Search query", i, j) ? (a.query ? esc(a.query) : `<span class="err">${esc(a.arguments)}</span>`) : "") :
                                     (shown("Read document", i, j) ? "read " + esc(a.docid) + (EV.has(pd(a.docid)) ? ' <span class="rk on">&#9733;</span>' : "") : "")).filter(Boolean).join("<br>");
      const a0 = t.actions[0];
      if (a0.type === "search" && shown("Search results", i, 0)) sub = a0.error ? `<div class="sub err">error</div>` : strip(a0);
    } else if ((t.calls || []).length && shown("Search query", i, 0)) {
      try { const g = JSON.parse(t.calls[0].arguments || "{}"); main = esc(g.user_query || g.docid || t.calls[0].arguments); } catch (e) { main = esc(t.calls[0].arguments); }
    }
    if (!main) main = `<span class="qq pend">reasoning&hellip;</span>`;
    rows.push(`<div class="row ${(st.sel ?? cur.i) === i ? "sel" : ""}" data-i="${i}"><span class="n">${i + 1}</span><span><div class="qq">${main}</div>${sub}</span></div>`);
  }
  if (cur.pend) rows.push(`<div class="row pend sel"><span class="n">${S.turns.length + 1}</span><span class="qq">thinking&hellip;</span></div>`);
  if (cur.fin) rows.push(`<div class="row sel" data-i="${S.turns.length}"><span class="n">&#8230;</span><span class="qq pend">writing the answer&hellip;</span></div>`);
  if (cur.ph === "Answer") rows.push(`<div class="row ans ${(st.sel ?? cur.i) === S.turns.length ? "sel" : ""}" data-i="${S.turns.length}"><span class="n">&check;</span><span class="qq">Answer: ${esc(S.answer.exact_answer || "(no exact answer)")}</span></div>`);
  const list = root.querySelector(".list");
  list.innerHTML = `<div class="cap"><b>Searches</b>one row per search: the query the agent wrote` +
    (EV.size ? `; the squares are its top 5 results, <span class="rk on" style="margin:0">gold</span> = a page with the answer` : `; the squares are its top 5 results`) + `</div>` + rows.join("");
  list.scrollTop = list.scrollHeight;
  list.querySelectorAll(".row[data-i]").forEach(el => el.addEventListener("click", () => { st.sel = +el.dataset.i; save(); paint(); }));

  // right: the selected turn, with only the parts already shown
  const det = root.querySelector(".det");
  const view = st.sel ?? cur.i;
  let h = "";
  if (view >= S.turns.length) {
    if (cur.pend) h = `<h4>Reasoning, turn ${S.turns.length + 1}</h4><div class="waiting">The model is reasoning&hellip;</div>`;
    else if (S.answer) {
      if (cur.ph === "Answer") {
        const a = S.answer, c = S.check;
        h += `<h4>Exact answer</h4><div class="exact">${esc(a.exact_answer || "(none)")}</div>`;
        if (c) h += c.correct === true ? `<div class="ok">Correct. Gold answer: ${esc(c.gold_answer)}</div>` :
          c.correct === false ? `<div class="bad">Wrong. Gold answer: ${esc(c.gold_answer)}</div>` : `<div>Not graded</div>`;
        h += `<h4>Final response</h4><div class="rs">${cites(a.text)}</div>`;
        if (a.reasoning) h += `<details><summary>final reasoning</summary><div class="rs">${esc(a.reasoning)}</div></details>`;
      } else h = `<h4>Reasoning before the answer</h4><div class="rs">${esc(S.answer.reasoning || "(no visible reasoning)")}</div>`;
    }
  } else {
    const t = S.turns[view];
    h = `<h4>Reasoning, turn ${view + 1}</h4><div class="rs">${esc(t.reasoning || "(no visible reasoning)")}</div>`;
    if (t.content) h += `<h4>Message</h4><div class="rs">${esc(t.content)}</div>`;
    t.actions.forEach((a, j) => {
      if (a.type === "search") {
        if (!shown("Search query", view, j)) return;
        if (a.error) { h += `<h4>Tool call</h4><div class="query">${esc(a.arguments)}</div><div class="err">error: ${esc(a.error)}</div>`; return; }
        h += `<h4 data-k="q${j}">Search query</h4><div class="query">${esc(a.query)}</div>`;
        if (a.q2q) h += `<div class="q2q"><span>Q2Q question</span>&nbsp; ${esc(a.q2q)}</div>`;
        if (!shown("Search results", view, j)) { h += `<div class="waiting">next: the search results</div>`; return; }
        h += `<h4 data-k="r${j}">Search results, top ${a.results.length}</h4><table>` + a.results.map((r, k) => { const rel = EV.has(D(r));
          return `<tr class="${rel ? "rel" : "irr"}"><td class="r"><span class="rk ${rel ? "on" : ""}">${k + 1}</span></td><td class="d">${esc(r.docid)}</td><td>` +
          `<b>${esc(r.title || "(untitled)")}</b>${r.date ? ` <span style="color:var(--mute)">${esc(r.date)}</span>` : ""}` +
          `${rel ? '<span class="badge">RELEVANT</span>' : ""}` +
          `<details><summary>text the agent sees (first 512 tokens)</summary><div class="snip">${esc(r.snippet)}</div></details></td></tr>`; }).join("") + `</table>`;
      } else if (a.type === "read") {
        if (!shown("Read document", view, j)) return;
        const rel = EV.has(pd(a.docid));
        h += `<h4 data-k="d${j}">Read document ${esc(a.docid)}${rel ? ' <span class="badge">RELEVANT</span>' : ""}</h4><div><b>${esc(a.title)}</b></div>` +
          `<details><summary>full text (${(a.text || "").length.toLocaleString()} characters)</summary><div class="snip">${esc((a.text || "").slice(0, 20000))}</div></details>`;
      }
    });
    if (!t.actions.length && (t.calls || []).length && shown("Search query", view, 0))
      h += `<h4>Tool call</h4>` + t.calls.map(c => `<div class="query">${esc(c.name)} ${esc(c.arguments)}</div>`).join("") + `<div class="waiting">searching&hellip;</div>`;
  }
  if (S.error && st.k >= N - 1) h = `<div class="err">${esc(S.error)}</div>` + h;
  det.innerHTML = `<div class="cap"><b>Current step</b>what the agent is thinking, the query it wrote, and what the search returned` +
    (view !== cur.i ? ` (an earlier turn; press Next to return)` : "") + `</div>` + h;
  const ck = view === cur.i && {"Search query": "q", "Search results": "r", "Read document": "d"}[cur.ph];
  const ce = ck && det.querySelector(`[data-k="${ck}${cur.j}"]`);
  const capH = (det.querySelector(".cap") || {}).offsetHeight || 0;
  det.scrollTop = ce ? Math.max(0, ce.offsetTop - capH - 8) : 0;     // keep the current step in view

  // relevant documents surfaced so far (up to the shown step)
  const first = {};
  S.turns.forEach((t, i) => t.actions.forEach((a, j) => { if (a.type === "search" && shown("Search results", i, j)) (a.results || []).forEach((r, k) => { if (EV.has(D(r)) && !(D(r) in first)) first[D(r)] = [i + 1, k + 1]; }); }));
  const cited = new Set(cur.ph === "Answer" ? ((S.answer.text || "").match(/\[[0-9a-z_, ]+\]/g)?.join(",").match(/[0-9a-z_]+/g) || []).map(pd) : []);
  root.querySelector(".ev").innerHTML = EV.size ? `<b>Relevant documents</b> (${Object.keys(first).length} of ${EV.size} found): ` +
    [...EV].map(d => first[d] ? `<span class="d hit">${d} · turn ${first[d][0]}, rank ${first[d][1]}${cited.has(d) ? " · cited" : ""}</span>` : `<span class="d miss">${d} · not found yet</span>`).join("") : "";
}
function cites(text) {
  return esc(text).replace(/\*\*([^*\n]+)\*\*/g, "<b>$1</b>")
    .replace(/(?:\[|\u3010)([0-9a-z_, ]+)(?:\]|\u3011)/g, (m, ids) => ids.split(/,\s*/).map(d => `<span class="cite ${EV.has(pd(d.trim())) ? "g" : ""}">${d.trim()}</span>`).join(" "));
}
paint();
})();
</script>
</div>
"""


class TrajectoryView:
    def __init__(self, question, qid=None, evidence_docids=(), config=None, show=True, context_limit=131072, step_mode=False):
        self.run = uuid.uuid4().hex[:8]
        self.state = {"run": self.run, "qid": qid, "question": question, "evidence": list(evidence_docids or []),
                      "context_limit": context_limit, "step_mode": bool(step_mode),
                      "config": {"model": "", "retriever": "", "tools": "search", **(config or {})},
                      "turns": [], "answer": None, "check": None, "pending": "model", "status": "running",
                      "elapsed": 0, "error": None, "speed": None}
        self.handle = display(HTML(self.render()), display_id=True) if (show and display) else None

    # agent_core.run_agent callback
    def __call__(self, kind, data):
        s = self.state
        s["elapsed"] = data.get("t", s["elapsed"])
        if kind == "start":
            s["config"].update({k: data[k] for k in ("model", "retriever") if data.get(k)})
            s["pending"] = "model"
        elif kind == "think":
            s["turns"].append({"reasoning": clean_reasoning(data.get("text", "")), "content": data.get("content", ""), "calls": data.get("calls", []),
                               "context_tokens": data.get("context_tokens"), "t": data.get("t"), "actions": []})
            s["pending"] = "tool"
        elif kind == "search":
            s["turns"][-1]["actions"].append({"type": "search", "query": data["query"], "results": data["results"],
                                              "arguments": data.get("arguments", ""), "error": data.get("error", ""),
                                              "q2q": data.get("q2q", ""), "t": data.get("t")})
            s["pending"] = "model"
        elif kind == "read":
            s["turns"][-1]["actions"].append({"type": "read", "docid": data["docid"], "title": data.get("title", ""),
                                              "text": data.get("text", ""), "t": data.get("t")})
            s["pending"] = "model"
        elif kind == "answer":
            last = {}
            if s["turns"] and not s["turns"][-1]["actions"] and not s["turns"][-1].get("calls"):
                last = s["turns"].pop()   # the final turn's reasoning is shown as the step before the answer
            s["answer"] = {"text": data.get("text", ""), "exact_answer": data.get("exact_answer", ""), "t": data.get("t"),
                           "reasoning": clean_reasoning(last.get("reasoning", "")), "context_tokens": last.get("context_tokens")}
            s["pending"], s["status"] = None, "done"
        elif kind == "error":
            s["error"], s["pending"], s["status"] = data.get("message"), None, "error"
        self.update()

    def set_check(self, check):
        self.state["check"] = check
        self.state["status"] = "done"
        self.update()

    def render(self):
        rid = "tv" + self.run + uuid.uuid4().hex[:4]
        data = json.dumps(self.state, ensure_ascii=False).replace("</", "<\\/")
        return TEMPLATE.replace("__ID__", rid).replace("__DATA__", data)

    def update(self):
        if self.handle is not None:
            self.handle.update(HTML(self.render()))


def replay(trace, speed=4.0, check=None, evidence_docids=None, show=True, step_mode=False):
    """Play a saved run (the 'steps' list from run_agent) through a TrajectoryView at `speed` times real time
    (speed=0: all at once, e.g. with step_mode=True so the presenter steps through it)."""
    steps = trace["steps"]
    start = next(s for s in steps if s["kind"] == "start")
    v = TrajectoryView(start["question"], qid=trace.get("qid"), evidence_docids=evidence_docids or trace.get("evidence_docids", []),
                       config={"model": start.get("model"), "retriever": start.get("retriever"),
                               "tools": "search, get_document" if trace.get("use_get_document") or "get_document" in str(trace.get("harness", "")) else "search"},
                       show=show, step_mode=step_mode)
    v.state["status"], v.state["speed"] = "replay", speed
    last = 0.0
    for s in steps:
        dt = max(0.0, (s.get("t", last) - last) / speed) if speed else 0.0
        if speed and show:
            time.sleep(min(dt, 8.0))
        last = s.get("t", last)
        v(s["kind"], s)
        if s["kind"] in ("answer", "error"):
            v.state["status"] = "done"
    if check or trace.get("check"):
        v.set_check(check or trace.get("check"))
    return v


# ---------------------------------------------------------------- side-by-side race of several configurations
RACE_TEMPLATE = r"""
<div id="__ID__" class="rv">
<style>
#__ID__{--ink:#1d1d1f;--mute:#6b6b70;--line:#d9d6cf;--paper:#fbfaf7;--blue:#1f5fbf;--gold:#11795a;--warn:#a5361b;
  font-family:Calibri,"Segoe UI",Arial,sans-serif;font-size:17px;color:var(--ink);background:var(--paper);border:1px solid var(--line);padding:14px 16px;max-width:1500px}
#__ID__ .qlab{font-size:13px;letter-spacing:.06em;text-transform:uppercase;color:var(--mute);font-weight:600;margin-bottom:4px}
#__ID__ .q{font-size:23px;font-weight:600;line-height:1.4;margin-bottom:20px;white-space:pre-wrap;max-width:1150px}
#__ID__ .bar{height:10px;background:#e7e4dc;border:1px solid var(--line);position:relative;margin:2px 0 10px}
#__ID__ .bar i{position:absolute;left:0;top:0;bottom:0;background:var(--blue)}
#__ID__ .bar i.warn{background:#c98a12} #__ID__ .bar i.full{background:var(--warn)}
#__ID__ .lanes{display:grid;grid-template-columns:repeat(__N__,1fr);gap:0;border-top:1px solid var(--line)}
#__ID__ .lane{border-right:1px solid var(--line);padding:8px 12px;min-width:0}
#__ID__ .lane:last-child{border-right:none}
#__ID__ .hd{font-size:21px;font-weight:700}
#__ID__ .ct{color:var(--mute);font-size:15.5px;margin:2px 0 8px}
#__ID__ .ct b{color:var(--ink);font-size:19px}
#__ID__ .list{height:430px;overflow:auto;border-top:1px solid var(--line)}
#__ID__ .row{display:grid;grid-template-columns:30px 1fr;padding:4px 0;border-bottom:1px solid #ece9e2}
#__ID__ .n{color:var(--mute)}
#__ID__ .qq{font-family:Menlo,Consolas,monospace;font-size:14px;word-break:break-word}
#__ID__ .g{color:#9a6a00;font-weight:700;font-size:13.5px}
#__ID__ .e{color:var(--warn);font-size:13.5px}
#__ID__ .rk{display:inline-block;min-width:19px;height:19px;line-height:17px;text-align:center;font-size:12px;margin:3px 3px 0 0;
  background:#fff;color:var(--mute);border:1px solid var(--line)}
#__ID__ .rk.on{background:#f2b705;border-color:#f2b705;color:#1d1d1f;font-weight:700}
#__ID__ .pend{color:var(--mute);font-style:italic;padding:4px 0 4px 30px}
#__ID__ .qz{font-size:14px;color:var(--mute)}
#__ID__ .cap2{font-size:14.5px;color:var(--mute);margin:-6px 0 12px;max-width:1150px}
#__ID__ .ans{margin-top:8px;font-size:19px;min-height:52px}
#__ID__ .ok{color:var(--gold);font-weight:700}
#__ID__ .sr{cursor:pointer}
#__ID__ .sr:hover .qq{text-decoration:underline}
#__ID__ .sr .rk{cursor:pointer}
#__ID__ .rk.sel{outline:2px solid var(--blue);outline-offset:1px}
#__ID__ .tw{color:var(--blue);font-size:12.5px;margin-left:4px}
#__ID__ .res{margin:6px 0 2px;border-left:3px solid var(--blue);padding:2px 0 2px 8px;background:#fff}
#__ID__ .ri{display:grid;grid-template-columns:24px 1fr;gap:4px;padding:3px 0;cursor:pointer;font-size:14.5px;line-height:1.3}
#__ID__ .ri:hover b{text-decoration:underline}
#__ID__ .ri.gold b{color:#7a5400}
#__ID__ .ri .d{font-family:Menlo,Consolas,monospace;font-size:12px;color:var(--mute)}
#__ID__ .snip{grid-column:1/3;white-space:pre-wrap;font-size:13.5px;line-height:1.35;max-height:220px;overflow:auto;background:#f6f4ef;border:1px solid var(--line);padding:6px 8px;margin:3px 0 4px;cursor:text}
#__ID__ .bad{color:var(--warn);font-weight:700}
</style>
<div class="qlab"></div><div class="q"><span class="qt"></span></div>
<div class="cap2">One column per setting, all running at the same time on the same question. One row per search: the query the agent wrote,
and its top 5 results as squares, <span class="rk on" style="margin:0">gold</span> = a page with the answer.
Click a row to see the 5 pages it returned, and click a page to read the text the agent got.</div>
<div class="lanes"></div>
<script>
(function(){
const S = __DATA__, root = document.getElementById("__ID__");
const esc = s => String(s ?? "").replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const EV = new Set(S.evidence || []), D = r => r.doc || r.docid;
const own = String(S.qid ?? "").startsWith("c");
const G = (window.__rv = window.__rv || {});
const UI = G[S.run] || (G[S.run] = {open: {}, doc: {}, scroll: {}, keep: {}});
root.querySelector(".qlab").textContent = S.qid === "audience" ? "Question from the audience" : own ? `Question ${S.qid} · written by us, answerable from the BrowseComp-Plus corpus` : `Question · BrowseComp-Plus #${S.qid ?? ""}`;
root.querySelector(".qt").textContent = S.question;
function resList(a, key) {
  return `<div class="res">` + (a.results || []).slice(0, 5).map((r, k) => {
    const g = EV.has(D(r)), on = UI.doc[key] === k;
    return `<div class="ri ${g ? "gold" : ""}" data-key="${key}" data-r="${k}"><span class="rk ${g ? "on" : ""}">${k + 1}</span>` +
      `<span><b>${esc(r.title || "(untitled)")}</b> <span class="d">${esc(r.docid)}${r.date ? " &middot; " + esc(r.date) : ""}</span></span>` +
      (on ? `<div class="snip">${esc(r.snippet || "(no text recorded)")}</div>` : "") + `</div>`;
  }).join("") + `</div>`;
}
function paint() {
  root.querySelector(".lanes").innerHTML = S.lanes.map((L, li) => {
    const rows = L.turns.map((t, i) => {
      const a = t.actions[0];
      if (a && a.type === "read") { const d = a.docid.startsWith("browsecomp_plus_") ? a.docid.slice(16).replace(/_[0-9]+$/, "") : a.docid;
        return `<div class="row"><span class="n">${i + 1}</span><span><span class="qq">read ${esc(a.docid)}</span>${EV.has(d) ? ' <span class="rk on">&#9733;</span>' : ""} ${esc(a.title || "")}</span></div>`; }
      if (!a) return (t.calls || []).length ? `<div class="row"><span class="n">${i + 1}</span><span class="qq">${esc((() => { try { return JSON.parse(t.calls[0].arguments).user_query || t.calls[0].arguments; } catch (e) { return t.calls[0].arguments; } })())}</span></div>` : "";
      if (a.error) return `<div class="row"><span class="n">${i + 1}</span><span><span class="qq">${esc(a.arguments)}</span><br><span class="e">error</span></span></div>`;
      const key = `${li}:${i}`, open = !!UI.open[key];
      const strip = (a.results || []).map((r, k) => `<span class="rk ${EV.has(D(r)) ? "on" : ""} ${open && UI.doc[key] === k ? "sel" : ""}" data-r="${k}">${k + 1}</span>`).join("");
      return `<div class="row sr" data-key="${key}"><span class="n">${i + 1}</span><span><span class="qq">${esc(a.query)}</span>` +
        (a.q2q ? `<br><span class="qz">Q2Q: ${esc(a.q2q)}</span>` : "") + `<br>${strip}<span class="tw">${open ? "&#9662; hide" : "&#9656; pages"}</span>` +
        (open ? resList(a, key) : "") + `</span></div>`;
    }).join("");
    const nS = L.turns.reduce((s, t) => s + t.actions.filter(a => a.type !== "read").length, 0);
    const nR = L.turns.reduce((s, t) => s + t.actions.filter(a => a.type === "read").length, 0);
    const found = new Set(); L.turns.forEach(t => t.actions.forEach(a => (a.results || []).forEach(r => { if (EV.has(D(r))) found.add(D(r)); })));
    let ans = "";
    if (L.answer) ans = `Answer: <b>${esc(L.answer.exact_answer || "(none)")}</b> ` + (L.check ? (L.check.correct ? '<span class="ok">correct</span>' : '<span class="bad">wrong</span>') : "");
    else if (L.error) ans = `<span class="bad">${esc(L.error)}</span>`;
    return `<div class="lane"><div class="hd">${esc(L.label)}</div>` +
      `<div class="ct">searches <b>${nS}</b>${nR ? ` &nbsp; documents read <b>${nR}</b>` : ""} &nbsp; time <b>${L.elapsed}s</b> &nbsp; relevant docs found <b>${found.size}/${EV.size}</b></div>` +
      `<div class="ct">context <b>${((L.ctx || 0) / 1000).toFixed(1)}k</b> of ${((S.context_limit || 131072) / 1000).toFixed(0)}k tokens</div>` +
      (() => { const p = Math.min(100, 100 * (L.ctx || 0) / (S.context_limit || 131072)); return `<div class="bar"><i class="${p > 85 ? "full" : p > 60 ? "warn" : ""}" style="width:${p.toFixed(1)}%"></i></div>`; })() +
      `<div class="list" data-li="${li}">${rows}${L.status === "running" ? `<div class="pend">${L.pending === "tool" ? "searching" : "thinking"}&hellip;</div>` : ""}</div>` +
      `<div class="ans">${ans}</div></div>`;
  }).join("");
  root.querySelectorAll(".list").forEach(l => {
    const li = l.dataset.li;
    l.scrollTop = UI.keep[li] ? (UI.scroll[li] || 0) : l.scrollHeight;
    l.addEventListener("scroll", () => { UI.scroll[li] = l.scrollTop; UI.keep[li] = l.scrollTop + l.clientHeight < l.scrollHeight - 4; });
  });
}
root.querySelector(".lanes").addEventListener("click", e => {
  if (e.target.closest(".snip")) return;
  const ri = e.target.closest(".ri"), row = e.target.closest(".sr");
  if (!row) return;
  const key = row.dataset.key, li = key.split(":")[0], lst = row.closest(".list");
  if (ri) { const k = +ri.dataset.r; UI.doc[key] = UI.doc[key] === k ? null : k; }
  else { const sq = e.target.closest(".rk[data-r]");
    if (sq) { const k = +sq.dataset.r; if (UI.open[key] && UI.doc[key] === k) { UI.doc[key] = null; } else { UI.open[key] = true; UI.doc[key] = k; } }
    else { UI.open[key] = !UI.open[key]; if (!UI.open[key]) UI.doc[key] = null; } }
  UI.keep[li] = true; UI.scroll[li] = lst ? lst.scrollTop : 0;
  paint();
});
paint();
})();
</script>
</div>
"""


class RaceView:
    """Several agent configurations on the same question, one column each, updated live from worker threads."""

    def __init__(self, question, qid=None, evidence_docids=(), labels=(), show=True):
        import threading
        self.lock = threading.Lock()
        self.state = {"run": uuid.uuid4().hex[:8], "qid": qid, "question": question, "evidence": list(evidence_docids or []), "context_limit": 131072,
                      "lanes": [{"label": l, "turns": [], "answer": None, "check": None, "status": "running",
                                 "pending": "model", "elapsed": 0, "error": None} for l in labels]}
        self.handle = display(HTML(self.render()), display_id=True) if (show and display) else None

    def lane(self, i):
        def cb(kind, data):
            with self.lock:
                L = self.state["lanes"][i]
                L["elapsed"] = data.get("t", L["elapsed"])
                if kind == "think":
                    L["turns"].append({"calls": data.get("calls", []), "actions": []})
                    L["pending"] = "tool"
                    L["ctx"] = data.get("context_tokens") or L.get("ctx")
                elif kind == "search":
                    L["turns"][-1]["actions"].append({"type": "search", "query": data["query"], "results": data["results"],
                                                      "arguments": data.get("arguments", ""), "error": data.get("error", ""),
                                                      "q2q": data.get("q2q", "")})
                    L["pending"] = "model"
                elif kind == "read":
                    L["turns"][-1]["actions"].append({"type": "read", "docid": data["docid"], "title": data.get("title", "")})
                    L["pending"] = "model"
                elif kind == "answer":
                    if L["turns"] and not L["turns"][-1]["actions"] and not L["turns"][-1]["calls"]:
                        L["turns"].pop()
                    L["answer"], L["status"] = {"exact_answer": data.get("exact_answer", "")}, "done"
                elif kind == "error":
                    L["error"], L["status"] = data.get("message"), "done"
                self.update()
        return cb

    def set_check(self, i, check):
        with self.lock:
            self.state["lanes"][i]["check"] = check
            self.update()

    def render(self):
        rid = "rv" + uuid.uuid4().hex[:10]
        data = json.dumps(self.state, ensure_ascii=False).replace("</", "<\\/")
        return RACE_TEMPLATE.replace("__ID__", rid).replace("__N__", str(len(self.state["lanes"]))).replace("__DATA__", data)

    def update(self):
        if self.handle is not None:
            self.handle.update(HTML(self.render()))


def race(qid, configs, llm, server, prompts, show=True, max_turns=60, grace=45):
    """Run the same question under several configs at once. configs: [{"label", "retriever", "model", ...}].
    When the first configuration answers, the others get `grace` more seconds, then stop."""
    import threading
    q = server.question(qid)
    view = RaceView(q["question"], qid=qid, evidence_docids=q["evidence_docids"], labels=[c["label"] for c in configs], show=show)
    results = [None] * len(configs)
    stop = {"at": None}

    def go(i, c):
        kw = {k: v for k, v in c.items() if k != "label"}
        results[i] = run_agent(q["question"], llm, server, prompts, on_event=view.lane(i), max_turns=max_turns,
                               deadline=lambda: stop["at"], **kw)
        if results[i]["response"] and stop["at"] is None:
            stop["at"] = time.time() + grace
        if results[i]["response"]:
            view.set_check(i, server.check(qid, results[i]["response"]))
    ts = [threading.Thread(target=go, args=(i, c)) for i, c in enumerate(configs)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()
    return view, results


# ---------------------------------------------------------------- the same query under different rankers, side by side
RANK_TEMPLATE = r"""
<div id="__ID__" class="kv">
<style>
#__ID__{--ink:#1d1d1f;--mute:#6b6b70;--line:#d9d6cf;--paper:#fbfaf7;--blue:#1f5fbf;--goldfill:#f2b705;--goldbg:#fff2c4;
  font-family:Calibri,"Segoe UI",Arial,sans-serif;font-size:17px;color:var(--ink);background:var(--paper);border:1px solid var(--line);padding:14px 16px;max-width:1500px}
#__ID__ *{box-sizing:border-box}
#__ID__ .qlab{font-size:13px;letter-spacing:.06em;text-transform:uppercase;color:var(--mute);font-weight:600;margin-bottom:6px}
#__ID__ .q{font-size:23px;font-weight:600;line-height:1.4;margin:0 0 16px;white-space:pre-wrap;max-width:1150px}
#__ID__ .cap{font-size:14.5px;color:var(--mute);margin:0 0 10px;max-width:1150px}
#__ID__ .lab{font-size:13px;letter-spacing:.05em;text-transform:uppercase;color:var(--mute);font-weight:600;margin-right:8px}
#__ID__ .query{display:inline-block;font-family:Menlo,Consolas,monospace;font-size:18px;color:var(--blue);background:#fff;border-left:4px solid var(--blue);padding:6px 12px;margin:0 0 14px}
#__ID__ .ctl{display:flex;gap:10px;align-items:center;margin:0 0 14px;font-size:15px;color:var(--mute)}
#__ID__ .ctl button{font:inherit;font-size:15.5px;padding:4px 14px;border:1px solid var(--line);background:#fff;color:var(--ink);cursor:pointer}
#__ID__ .ctl button.main{background:var(--blue);border-color:var(--blue);color:#fff;font-weight:700}
#__ID__ .ctl button:disabled{opacity:.45;cursor:default}
#__ID__ .cols{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:0;border-top:1px solid var(--line)}
#__ID__ .col{border-right:1px solid var(--line);padding:10px 12px 12px;min-width:0}
#__ID__ .col:last-child{border-right:none}
#__ID__ .hd{font-size:20px;font-weight:700;line-height:1.25}
#__ID__ .sub{font-size:14px;color:var(--mute);margin:3px 0 0;min-height:19px}
#__ID__ .qz{font-size:15px;color:var(--ink);background:#fff;border-left:4px solid var(--mute);padding:4px 8px;margin:6px 0 0}
#__ID__ .qz span{display:block;font-size:12.5px;color:var(--mute);text-transform:uppercase;letter-spacing:.04em}
#__ID__ ol{list-style:none;margin:10px 0 0;padding:0}
#__ID__ li{display:grid;grid-template-columns:28px 1fr;gap:6px;padding:6px 4px;border-bottom:1px solid #ece9e2}
#__ID__ li.rel{background:var(--goldbg);box-shadow:inset 5px 0 0 var(--goldfill)}
#__ID__ .rk{display:inline-block;min-width:22px;height:22px;line-height:20px;text-align:center;font-size:13px;background:#fff;color:var(--mute);border:1px solid var(--line)}
#__ID__ li.rel .rk{background:var(--goldfill);border-color:var(--goldfill);color:#1d1d1f;font-weight:700}
#__ID__ .t{font-size:15.5px;line-height:1.3;display:-webkit-box;-webkit-line-clamp:3;-webkit-box-orient:vertical;overflow:hidden}
#__ID__ li.rel .t{font-weight:700}
#__ID__ .d{font-family:Menlo,Consolas,monospace;font-size:12.5px;color:var(--mute)}
#__ID__ .foot{margin-top:10px;font-size:16px}
#__ID__ .foot b{font-size:19px}
#__ID__ .foot .ok{color:#9a6a00;font-weight:700}
#__ID__ .wait{color:var(--mute);font-style:italic;margin-top:12px}
#__ID__ .hidden{opacity:.12;filter:blur(1.5px)}
</style>
<div class="qlab"></div><div class="q"></div>
<div class="cap">Same question, same query, written by the agent in one of its runs; only the ranking changes. Each column shows the
top 5 results the agent would get, <span class="rk" style="background:#f2b705;border-color:#f2b705;color:#1d1d1f;font-weight:700">&nbsp;</span> = <span class="evword">a page with the answer</span>.</div>
<div><span class="lab">Agent query</span><span class="query"></span></div>
<div class="ctl"></div>
<div class="cols"></div>
<script>
(function(){
const S = __DATA__, ROOT_ID = "__ID__";
const esc = s => String(s ?? "").replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const EV = new Set(S.evidence || []);
const EVW = String(S.qid ?? "").startsWith("c") ? "page with the answer" : "evidence page";
const G = (window.__kv = window.__kv || {});
const st = G[S.run] || (G[S.run] = {k: S.step_mode ? 1 : S.cols.length});
function act(w) { if (w === "next") st.k = Math.min(S.cols.length, st.k + 1); else if (w === "back") st.k = Math.max(1, st.k - 1); else st.k = S.cols.length; paint(); }
window.__tvAct = window.__tvAct || {}; window.__tvAct[S.run] = act;
if (S.step_mode) window.__tvActive = S.run;
function paint() {
  const root = document.getElementById(ROOT_ID); if (!root) return;
  root.classList.add("tv"); root.dataset.run = S.run;
  root.querySelector(".qlab").textContent = String(S.qid ?? "").startsWith("c") ? `Question ${S.qid} · written by us, answerable from the BrowseComp-Plus corpus` : `Question · BrowseComp-Plus #${S.qid}`;
  root.querySelector(".q").textContent = S.question;
  root.querySelector(".query").textContent = S.query;
  if (!String(S.qid ?? "").startsWith("c")) root.querySelector(".evword").textContent = "a gold evidence page (one of the pages that together give the answer)";
  root.querySelector(".ctl").innerHTML = `<button data-a="back" ${st.k <= 1 ? "disabled" : ""}>&#9664; Back</button>` +
    `<button class="main" data-a="next" ${st.k >= S.cols.length ? "disabled" : ""}>Next ranker &#9654;</button>` +
    `<button data-a="all">Show all</button><span>${st.k} of ${S.cols.length} shown</span>`;
  root.querySelectorAll(".ctl button").forEach(b => b.addEventListener("click", () => { window.__tvActive = S.run; act(b.dataset.a); }));
  root.querySelector(".cols").innerHTML = S.cols.map((c, i) => {
    let body;
    if (!c.results) body = `<div class="wait">ranking&hellip;</div>`;
    else if (c.error) body = `<div class="wait">${esc(c.error)}</div>`;
    else body = (c.q2q ? `<div class="qz"><span>Q2Q question</span>${esc(c.q2q)}</div>` : "") +
      `<ol>` + c.results.slice(0, 5).map((r, k) => { const rel = EV.has(r.doc || r.docid);
        return `<li class="${rel ? "rel" : ""}"><span class="rk">${k + 1}</span><span><span class="t">${esc(r.title || "(untitled)")}</span><span class="d">${esc(r.docid)}</span></span></li>`; }).join("") + `</ol>` +
      `<div class="foot">${c.rel_rank ? (c.rel_rank <= 5 ? `first ${EVW} at <b class="ok">rank ${c.rel_rank}</b>` : `first ${EVW} at rank <b>${c.rel_rank}</b>, not shown to the agent`) : `${EVW} <b>not in the top ${c.depth || 20}</b>`}</div>`;
    return `<div class="col ${i >= st.k ? "hidden" : ""}"><div class="hd">${esc(c.label)}</div><div class="sub">${esc(c.note || "")}</div>${body}</div>`;
  }).join("");
}
paint();
})();
</script>
</div>
"""


class RankerView:
    """One agent query, the top 5 under several rankers side by side; step_mode reveals one column per Next."""

    def __init__(self, question, qid, evidence_docids, query, settings, show=True, step_mode=True):
        self.state = {"run": uuid.uuid4().hex[:8], "qid": qid, "question": question, "evidence": list(evidence_docids or []),
                      "query": query, "step_mode": bool(step_mode),
                      "cols": [{"label": s["label"], "note": s.get("note", ""), "retriever": s["retriever"]} for s in settings]}
        self.handle = display(HTML(self.render()), display_id=True) if (show and display) else None

    def set(self, i, results=None, q2q="", error="", depth=20):
        c = self.state["cols"][i]
        ev = set(self.state["evidence"])
        c.update(results=results or [], q2q=q2q, error=error, depth=depth,
                 rel_rank=next((k + 1 for k, r in enumerate(results or []) if (r.get("doc") or r["docid"]) in ev), None))
        if self.handle is not None:
            self.handle.update(HTML(self.render()))

    def render(self):
        rid = "kv" + uuid.uuid4().hex[:10]
        return RANK_TEMPLATE.replace("__ID__", rid).replace("__DATA__", json.dumps(self.state, ensure_ascii=False).replace("</", "<\\/"))


# Chuan 2026-11-06: one BM25 column, with the best parameters from the SIGIR 2026 paper (k1 10, b 1), labelled just "BM25"
RANKERS = [{"label": "BM25", "note": "matches words", "retriever": "bm25"},
           {"label": "ColBERTv2", "note": "neural retriever, matches meaning", "retriever": "colbert"},
           {"label": "BM25 → monoT5", "note": "re-ranks the BM25 top 50", "retriever": "bm25+monot5"}]
# the Q2Q column ("bm25+monot5+q2q", both retrieval and re-ranking see the rewritten question) was dropped from the final demo on
# 2026-11-06; the server still supports it, and archive_with_q2q_2026-11-05/ keeps the version that showed it


def compare_rankers(qid, query, reasoning, server, settings=None, show=True, step_mode=True, recorded=None, depth=20):
    """Run one agent query through several rankers (live), or show recorded results ({retriever: {"results", "q2q"}})."""
    settings = settings or RANKERS
    q = server.question(qid) if server is not None else recorded["_question"]
    view = RankerView(q["question"], qid, q["evidence_docids"], query, settings, show=show, step_mode=step_mode)
    out = {"_question": {"question": q["question"], "evidence_docids": q["evidence_docids"]}}
    for i, s in enumerate(settings):
        if recorded is not None:
            d = recorded.get(s["retriever"], {})
            view.set(i, d.get("results"), d.get("q2q", ""), "" if d else "not recorded", depth)
            continue
        try:
            d = server.search(query, retriever=s["retriever"], k=depth, reasoning=reasoning)
            view.set(i, d["results"], d.get("q2q", ""), depth=depth)
            out[s["retriever"]] = {"results": d["results"], "q2q": d.get("q2q", "")}
        except Exception as e:  # noqa: BLE001
            view.set(i, [], "", f"error: {str(e)[:120]}", depth)
    return out

_race = race  # private alias, so a notebook variable named race cannot break compare()

# ---------------------------------------------------------------- recorded runs (replay mode)
# BrowseComp-Plus asks that its questions and answers never appear as plain text online, so the recorded runs are
# stored obfuscated and decoded here at run time (the same idea as the benchmark's own encrypted release).
TRACES_URL = "https://agentic-search.github.io/cikm2026/demo/traces.enc"
TRACES_MIRROR = "https://raw.githubusercontent.com/agentic-search/agentic-search.github.io/main/cikm2026/demo/traces.enc"
_KEY = b"agentic-search-tutorial-cikm2026-browsecomp-plus-demo"


def _decode(blob):
    data = base64.b64decode(blob)
    stream, i = b"", 0
    while len(stream) < len(data):
        stream += hashlib.sha256(_KEY + i.to_bytes(4, "big")).digest()
        i += 1
    return json.loads(bytes(a ^ b for a, b in zip(data, stream)).decode("utf-8"))


TRACES = {}


def load_traces(path_or_url=TRACES_URL):
    global TRACES
    if os.path.exists(path_or_url):
        TRACES = _decode(open(path_or_url).read())
        return TRACES
    err = None
    for url in dict.fromkeys([path_or_url, TRACES_MIRROR]):    # the website first, then the same file on GitHub
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            TRACES = _decode(r.text)
            return TRACES
        except Exception as e:  # noqa: BLE001
            err = e
    raise err


def _trace_key(qid, retriever="bm25", use_get_document=False):
    return f"{qid}|{retriever}|{'getdoc' if use_get_document else 'search'}"


# ---------------------------------------------------------------- connecting
SERVER = LLM = None
PROMPTS = {}
MODE = "replay"


def ensure_ready():
    """Replay mode needs the recorded runs; load them here if the Connect cell has not been run."""
    global PROMPTS
    if MODE == "replay" and not TRACES:
        load_traces()
    if not PROMPTS:
        PROMPTS = TRACES.get("_prompts", {})


def connect(search_url="", search_key="", llm_base_url="", llm_api_key="EMPTY"):
    """LIVE mode if both of your endpoints answer, REPLAY mode otherwise."""
    global SERVER, LLM, PROMPTS, MODE
    MODE = "replay"
    reason = "no endpoints were entered"
    if search_url and llm_base_url:
        try:
            h = requests.get(search_url.rstrip("/") + "/health", timeout=10).json()
            assert h.get("ok"), h
            server = DemoServer(search_url, search_key, timeout=600, local_questions=globals().get("LOCAL_QUESTIONS"))
            llm = OpenAI(base_url=llm_base_url.rstrip("/"), api_key=llm_api_key or "EMPTY", timeout=900, max_retries=2)
            try:
                models = [m.id for m in llm.models.list().data]
            except Exception:  # noqa: BLE001
                models = []
            try:
                prompts = server.prompts()                      # optional route; otherwise use the official templates below
            except Exception:  # noqa: BLE001
                prompts = load_traces().get("_prompts", {})
            SERVER, LLM, PROMPTS, MODE = server, llm, prompts, "live"
            display(Markdown(f"**LIVE**: connected to your endpoints. Models: {', '.join(models) or 'unknown'}."))
            return
        except Exception as e:  # noqa: BLE001
            reason = f"the endpoints could not be reached ({type(e).__name__}: {str(e)[:150]})"
    elif search_url or llm_base_url:
        reason = "live mode needs both the model endpoint and the search endpoint"
    try:
        load_traces()
    except Exception as e2:  # noqa: BLE001
        display(Markdown(f"**Not connected.** {reason[0].upper() + reason[1:]}, and the recorded runs could not be downloaded "
                         f"({type(e2).__name__}). Check that you can open {TRACES_URL}."))
        return
    PROMPTS = TRACES.get("_prompts", {})
    display(Markdown(f"**REPLAY** ({reason}): the cells below replay runs we recorded in advance "
                     f"({len([k for k in TRACES if not k.startswith('_')])} runs). The viewer and the code are the same as in the live demo."))


In [2]:
#@title Helpers to run (or replay) a question { display-mode: "form" }
RETRIEVER_LABEL = {"bm25": "BM25", "colbert": "ColBERTv2", "bm25+monot5": "BM25 + monoT5 re-ranker"}
# our demo question (graded by word match)
LOCAL_QUESTIONS = {"c16": {"question": "Notre-Dame de Paris reopened after the 2019 fire. On what date did it reopen, and which pop star performed at the reopening?", "answer": "December 7 Pharrell", "aliases": ["7 December Pharrell", "Dec 7 Pharrell"], "display_answer": "7 December 2024; Pharrell Williams", "evidence_docids": ["18237", "40032", "73379"]}}
# one query the agent wrote for this question in a recorded run, used in Sections 2-3
LAB_QUERY = {"qid": "c16", "query": "Notre-Dame de Paris reopened date after 2019 fire pop star performed at reopening", "reasoning": ""}


def demo(qid, retriever="bm25", use_get_document=False, model="gpt-oss-20b", reasoning_effort="high", step=True, speed=3.0,
         max_minutes=3):
    """Run one question with the agent above and watch it. step=True: move through the steps yourself with Next/Back
    (or the arrow keys) while the agent keeps working; press Live to follow the agent instead. The run stops after
    max_minutes so the next cell can start."""
    ensure_ready()
    if MODE == "replay":
        t = TRACES.get(_trace_key(qid, retriever, use_get_document))
        if t is None:
            print("No recorded run for this setting. Recorded:", [k for k in TRACES if not k.startswith("_")])
            return None
        replay(t, speed=0 if step else speed, step_mode=step)
        return t
    q = SERVER.question(qid)
    view = TrajectoryView(q["question"], qid=qid, evidence_docids=q["evidence_docids"], step_mode=step,
                          config={"model": model, "retriever": retriever,
                                  "tools": "search, get_document" if use_get_document else "search"})
    end = time.time() + 60 * max_minutes
    result = run_agent(q["question"], LLM, SERVER, PROMPTS, model=model, retriever=retriever,
                       reasoning_effort=reasoning_effort, use_get_document=use_get_document, on_event=view,
                       deadline=lambda: end)
    if result["response"]:
        check = SERVER.check(qid, result["response"])
        view.set_check(check)
        result["check"] = check
    result.update(qid=qid, evidence_docids=q["evidence_docids"])
    return result


def ask(question, retriever="bm25+monot5", use_get_document=True, model="gpt-oss-20b", reasoning_effort="high",
        max_minutes=4, step=True):
    """Run the agent on any question. There is no gold answer, so nothing is graded; the run stops after max_minutes."""
    question = " ".join(str(question or "").split())
    if MODE != "live":
        ensure_ready()
        t = TRACES.get("_audience")
        if t:
            display(Markdown(f"**Replay**: a question from the audience at the tutorial, *{t['steps'][0]['question']}*, recorded "
                             f"with BM25 + monoT5 and `get_document`. To ask your own question, enter your endpoints in the *Connect* cell."))
            replay(t, speed=0 if step else 3.0, step_mode=step)
            return t
        display(Markdown("This part needs live mode: enter your own model and search endpoints in the *Connect* cell."))
        return None
    if not question:
        print("Paste a question into the QUESTION box above, then run the cell again.")
        return None
    view = TrajectoryView(question, qid="audience", evidence_docids=[], step_mode=step,
                          config={"model": model, "retriever": retriever,
                                  "tools": "search, get_document" if use_get_document else "search"})
    end = time.time() + 60 * max_minutes
    return run_agent(question, LLM, SERVER, PROMPTS, model=model, retriever=retriever, reasoning_effort=reasoning_effort,
                     use_get_document=use_get_document, on_event=view, max_turns=40, deadline=lambda: end)


def rankers(settings, step=True):
    """One recorded agent query on our question under several rankers, side by side (recorded results in replay mode)."""
    pick = [r for r in RANKERS if r["retriever"] in settings]
    if MODE == "replay":
        lab, rec = TRACES.get("_lab"), TRACES.get("_rankers")
        if not lab or not rec:
            print("No recorded ranking in the replay bundle.")
            return None
        return compare_rankers(lab["qid"], lab["query"], lab["reasoning"], None, pick, step_mode=step, recorded=rec)
    return compare_rankers(LAB_QUERY["qid"], LAB_QUERY["query"], LAB_QUERY["reasoning"], SERVER, pick, step_mode=step)


def sigir(title, rows, note=""):
    """Bar chart of numbers from our SIGIR 2026 paper. rows: [(label, accuracy in %, highlight)]."""
    bars = "".join(f'<tr><td style="padding:5px 12px;text-align:right">{n}</td><td style="width:560px"><div style="background:{"#f2b705" if hi else "#8a8f98"};'
                   f'height:24px;width:{v * 8:.0f}px;display:inline-block;vertical-align:middle"></div> <b>{v:.1f}%</b></td></tr>' for n, v, hi in rows)
    display(HTML(f'<div style="font-family:Calibri,Arial,sans-serif;font-size:18px;color:#1d1d1f"><b>{title}</b>'
                 f'<table style="border-collapse:collapse;margin-top:8px">{bars}</table>'
                 f'<div style="color:#6b6b70;font-size:15px;margin-top:6px">{note}</div></div>'))


def compare(qid, configs, speed=3.0, grace=45):
    """Run (or replay) the same question under several settings side by side. Once one setting answers,
    the others get `grace` more seconds and then stop."""
    global LAST_COMPARE
    ensure_ready()
    if MODE == "live":
        LAST_COMPARE = _race(qid, configs, LLM, SERVER, PROMPTS, grace=grace)[1]
        return
    traces = [TRACES.get(_trace_key(qid, c.get("retriever", "bm25"), c.get("use_get_document", False))) for c in configs]
    if not all(traces):
        print("Not all settings were recorded for this question.")
        return None
    first = next(s for s in traces[0]["steps"] if s["kind"] == "start")
    view = RaceView(first["question"], qid=qid, evidence_docids=traces[0].get("evidence_docids", []), labels=[c["label"] for c in configs])

    stop = {"at": None}

    def play(i, t):
        last = 0.0
        for s in t["steps"]:
            time.sleep(min(max(0.0, (s.get("t", last) - last) / speed), 8.0))
            last = s.get("t", last)
            if stop["at"] and time.time() > stop["at"] and s["kind"] not in ("answer", "error"):
                n = sum(1 for x in t["steps"] if x["kind"] == "search" and x.get("t", 0) <= last)
                total = sum(1 for x in t["steps"] if x["kind"] == "search")
                ok = (t.get("check") or {}).get("correct")
                end = f"the full run went on to {total} searches and {'a correct' if ok else 'a wrong' if ok is False else 'no'} answer"
                view.lane(i)("error", {"message": f"stopped by us after {n} searches ({end})", "t": last})
                return
            view.lane(i)(s["kind"], s)
            if s["kind"] == "answer" and stop["at"] is None:
                stop["at"] = time.time() + grace / speed
        if t.get("check"):
            view.set_check(i, t["check"])
    ts = [threading.Thread(target=play, args=(i, t)) for i, t in enumerate(traces)]
    for t in ts:
        t.start()
    for t in ts:
        t.join()


In [3]:
#@title Connect: run this cell, with the fields left empty to replay our recorded runs { display-mode: "form" }
#@markdown For live mode, enter your own endpoints (see "How to use this notebook" at the top).
LLM_BASE_URL = ""  #@param {type:"string"}
LLM_API_KEY = "EMPTY"  #@param {type:"string"}
SEARCH_URL = ""  #@param {type:"string"}
SEARCH_API_KEY = ""  #@param {type:"string"}
connect(SEARCH_URL, SEARCH_API_KEY, llm_base_url=LLM_BASE_URL, llm_api_key=LLM_API_KEY)


**REPLAY** (no endpoints were entered): the cells below replay runs we recorded in advance (4 runs). The viewer and the code are the same as in the live demo.

## The agent (our setup)
The agent is a plain loop: the model reads the question and everything so far, reasons, and either writes a search query or answers; we run the search, append the top 5 results, and repeat. Nothing is trained or tuned: an off-the-shelf model (gpt-oss-20b) and an off-the-shelf retriever (BM25).

In [4]:
#@title The two tools the agent can call, and its prompt { display-mode: "form" }
# Official BrowseComp-Plus definitions: search returns the top 5 documents with their first 512 tokens; get_document returns a
# whole document and is switched off until Section 4. The full JSON definitions and the prompt are folded; click to open.
import html as _h
ensure_ready()
_fold = lambda title, body: ("<details style='margin:6px 0'><summary style='cursor:pointer;color:#555'>" + title + "</summary>"
                             "<pre style='white-space:pre-wrap;font-size:13px'>" + _h.escape(body) + "</pre></details>")
display(HTML("<div style='font-size:18px;line-height:1.7'>"
             "<code><b>search(user_query)</b></code> &rarr; the top 5 documents, first 512 tokens of each<br>"
             "<code><b>get_document(docid)</b></code> &rarr; a whole document (switched off until Section 4)</div>"
             + _fold("Full tool definitions (JSON)", json.dumps([SEARCH_TOOL["function"], GET_DOCUMENT_TOOL["function"]], indent=2))
             + _fold("The prompt (official BrowseComp-Plus template, search only)",
                     PROMPTS.get("QUERY_TEMPLATE_NO_GET_DOCUMENT", "(loaded in live mode)"))))


The loop itself. `on_event` is how the viewer sees each step.

In [5]:
def run_agent(question, llm, server, prompts, model="gpt-oss-20b", retriever="bm25", reasoning_effort="high",
              use_get_document=False, max_turns=100, keep_reasoning=True, on_event=None, max_tokens=16384, deadline=None):
    """Run one question. Returns a dict with the final response, every step, and counts. on_event(kind, data) is
    called after each step: kind in {"start", "think", "search", "read", "answer", "error"}."""
    emit = on_event or (lambda kind, data: None)
    template = prompts["QUERY_TEMPLATE" if use_get_document else "QUERY_TEMPLATE_NO_GET_DOCUMENT"]
    messages = [{"role": "user", "content": template.format(Question=question)}]
    tools = [SEARCH_TOOL] + ([GET_DOCUMENT_TOOL] if use_get_document else [])
    t0, steps, seen = time.time(), [], []
    n_search = n_doc = 0
    final, prompt_tokens, completion_tokens = "", [], 0

    def log(kind, **data):
        data["t"] = round(time.time() - t0, 1)
        steps.append({"kind": kind, **data})
        emit(kind, data)

    log("start", question=question, model=model, retriever=retriever)
    for turn in range(max_turns):
        if deadline and deadline() and time.time() > deadline():
            log("error", message=f"stopped by us after {n_search} searches, no answer yet")
            break
        r, err = None, None
        for attempt in range(3):          # a failed call is retried, so one bad sample does not end a live run
            try:
                r = llm.chat.completions.create(model=model, messages=messages, tools=tools, max_tokens=max_tokens,
                                                reasoning_effort=reasoning_effort)
                if (r.choices[0].message.content or "").strip() or r.choices[0].message.tool_calls:
                    break
                err, r = RuntimeError("the model returned neither text nor a tool call"), None   # resample an empty reply
            except Exception as e:  # noqa: BLE001
                err = e
                time.sleep(1 + attempt)
        if r is None:
            log("error", message=str(err)[:500])
            break
        m = r.choices[0].message
        for tc in m.tool_calls or []:
            tc.function.name = clean_tool_name(tc.function.name, [t["function"]["name"] for t in tools])
        extra = m.model_extra or {}
        reasoning = extra.get("reasoning") or extra.get("reasoning_content") or ""
        if r.usage:
            prompt_tokens.append(r.usage.prompt_tokens)
            completion_tokens += r.usage.completion_tokens
        calls = [{"name": tc.function.name, "arguments": tc.function.arguments} for tc in (m.tool_calls or [])]
        log("think", text=reasoning, content=m.content or "", calls=calls,
            context_tokens=r.usage.prompt_tokens if r.usage else None)
        am = {"role": "assistant", "content": m.content or ""}
        if reasoning and keep_reasoning:
            am["reasoning"] = reasoning
        if m.tool_calls:
            am["tool_calls"] = [tc.model_dump(exclude_none=True) for tc in m.tool_calls]
        messages.append(am)
        if not m.tool_calls:
            final = m.content or ""
            log("answer", text=final, exact_answer=exact_answer(final))
            break
        for tc in m.tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except ValueError:
                args = {}
            if tc.function.name == "search":
                q = str(args.get("user_query") or args.get("query") or "")
                err = ""
                try:
                    reply = server.search(q, retriever=retriever, reasoning=reasoning if retriever.endswith("+q2q") else "")
                    res = reply["results"]
                    out = json.dumps([{"docid": x["docid"], "score": x["score"], "snippet": x["snippet"]} for x in res], indent=2)
                except Exception as e:  # noqa: BLE001
                    err = str(e)[:300]
                    reply, res, out = {}, [], json.dumps({"error": err})
                n_search += 1
                seen += [x.get("doc", x["docid"]) for x in res if x.get("doc", x["docid"]) not in seen]
                log("search", query=q, arguments=tc.function.arguments, error=err, q2q=reply.get("q2q", ""),
                    results=[{k: x.get(k) for k in ("docid", "doc", "score", "title", "date", "snippet")} for x in res])
            elif tc.function.name == "get_document":
                d = str(args.get("docid", ""))
                try:
                    doc = server.get_document(d)
                    out = json.dumps({"docid": doc["docid"], "text": doc["text"]}, indent=2)
                except Exception as e:  # noqa: BLE001
                    doc, out = {"docid": d, "text": "", "title": ""}, json.dumps({"error": str(e)[:300]})
                n_doc += 1
                log("read", docid=d, title=doc.get("title", ""), text=doc.get("text", ""))
            else:
                out = json.dumps({"error": f"unknown tool {tc.function.name}"})
                log("search", query="", arguments=f"{tc.function.name}({tc.function.arguments})", error=f"unknown tool {tc.function.name}", results=[])
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    else:
        log("error", message=f"stopped after {max_turns} turns without an answer")
    return {"response": final, "exact_answer": exact_answer(final), "steps": steps, "messages": messages,
            "searches": n_search, "documents": n_doc, "seen_docids": seen, "secs": round(time.time() - t0, 1),
            "context_tokens": prompt_tokens, "completion_tokens": completion_tokens, "model": model, "retriever": retriever}


## 1. One question, step by step
The box at the top is the agent loop. On the left is **the LLM**, which reasons and then either calls a tool or answers. On the right are **the tools**, which run outside the LLM on the search service: `search` now, and `get_document` (greyed out) from Section 4. The current step is lit in blue. Press **Next ▶** (or the right arrow key) to move on; the agent keeps working in the background, and **Live** follows it. The left column has one row per search; the squares under each query are its top 5 results, **gold** when the page contains the answer. Click any row to look at it again. The bar shows how much of the model's context the run has used.

Note what the agent can see: only the first 512 tokens of each result.

*Before we start: will it get the answer?*

In [6]:
QID = "c16"      # our question, used in Sections 1-4
result = demo(QID)

## 2. Change the retriever
Same question, two agents at the same time. Everything is the same except the search engine behind the `search` tool: BM25 matches words; ColBERTv2 is a neural retriever that matches meaning. Gold marks a page with the answer. Watch which one finds it first.

In [7]:
runs = compare(QID, [{"label": "BM25", "retriever": "bm25"},
                     {"label": "ColBERTv2", "retriever": "colbert"}])

## 3. Add a re-ranker
Same question, two agents at the same time. On the right, BM25 retrieves the top 50 documents and a neural re-ranker (monoT5) reads each of them with the query and puts the best 5 first. The agent still sees only 5 results.

In [8]:
runs = compare(QID, [{"label": "BM25", "retriever": "bm25"},
                     {"label": "BM25 + monoT5 re-ranker", "retriever": "bm25+monot5"}])

## 4. Let the agent read whole documents
Same question again, run twice at the same time: search only, and search plus a second tool, `get_document`, which returns a whole page. Watch whether the agent opens a page and whether that changes its searches or its answer.

In [9]:
runs = compare(QID, [{"label": "search only", "retriever": "bm25"},
                     {"label": "search + get_document", "retriever": "bm25", "use_get_document": True}])

## 5. Your question
Ask the agent anything. This needs live mode, with your own endpoints; in replay mode the cell plays a question from the audience at the tutorial. The corpus is a fixed set of about 100K web pages from BrowseComp-Plus, and we have not checked your question against it. We go through what the agent does: the queries it writes, what it finds, and how it answers.

In [10]:
#@title Ask the agent a question from the audience { display-mode: "form" }
#@markdown Paste the question into the box, then run the cell.
QUESTION = ""  #@param {type:"string"}
#@markdown ---
RETRIEVER = "bm25+monot5"  #@param ["bm25", "colbert", "bm25+monot5"]
READ_WHOLE_DOCUMENTS = True  #@param {type:"boolean"}
result = ask(QUESTION, retriever=RETRIEVER, use_get_document=READ_WHOLE_DOCUMENTS)


This part needs live mode: enter your own model and search endpoints in the *Connect* cell.

## What we saw
* The agent breaks the question into steps, rewrites its queries after reading the results, and answers when it can cite a page.
* The retriever decides what the agent gets to see: with the same model and prompt, a different retriever leads to different searches and sometimes a different answer.
* A re-ranker can lift the page with the answer into the 5 results the agent reads.
* Reading whole documents lets the agent use what the first 512 tokens of a page leave out.

Part II continues with long-horizon agents and evaluation.